In [ ]:
"""
League of Legends Patch Notes Scraper — 2026 Season
Outputs structured JSON + JSONL ready for RAG ingestion.

Install: pip install requests beautifulsoup4
Run:     python lol_patch_scraper.py
"""

import requests
from bs4 import BeautifulSoup
import json, time, re, os
from collections import Counter

OUTPUT_DIR  = "patch_notes"
DELAY       = 1.5
BASE_DOMAIN = "https://www.leagueoflegends.com"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
}

def normalize_section(name):
    n = name.lower()
    if "bugfix" in n or "quality of life" in n or "qol" in n:
        return "Bugfixes & QoL"
    if "summoner" in n:
        return "Summoner Spells"
    if "arena" in n and "champion" in n:
        return "Champions (Arena)"
    if "arena" in n and "item" in n:
        return "Items (Arena)"
    if "champion" in n:
        return "Champions"
    if "item" in n:
        return "Items"
    if "rune" in n:
        return "Runes"
    return name

def get_all_patch_urls():
    patch_urls = []
    seen = set()
    print("  Probing patch-26-1 through patch-26-15 (both URL patterns)...")
    for i in range(1, 16):
        candidates = [
            f"{BASE_DOMAIN}/en-us/news/game-updates/patch-26-{i}-notes/",
            f"{BASE_DOMAIN}/en-us/news/game-updates/league-of-legends-patch-26-{i}-notes/",
        ]
        for url in candidates:
            if url in seen:
                continue
            try:
                r = requests.head(url, headers=HEADERS, timeout=10, allow_redirects=True)
                if r.status_code == 200:
                    patch_urls.append(url)
                    seen.add(url)
                    print(f"    ✓ patch-26-{i} found: {url.rstrip('/').split('/')[-1]}")
                    break
                time.sleep(0.3)
            except Exception as e:
                print(f"    ✗ error: {e}")
    return patch_urls


def parse_change_block(div):
    name_tag = div.find(["h3", "h2"], class_=re.compile("change-title"))
    if not name_tag:
        return None
    name    = name_tag.get_text(strip=True)
    ctx_tag = div.find("blockquote", class_=re.compile("context"))
    context = ctx_tag.get_text(strip=True) if ctx_tag else ""
    changes = []
    for h4 in div.find_all("h4", class_=re.compile("change-detail-title")):
        ability = re.sub(r"\s+", " ", h4.get_text(strip=True)).strip()
        bullets = []
        sib = h4.find_next_sibling()
        while sib and sib.name not in ("h4", "h2", "h3", "hr"):
            if sib.name == "ul":
                for li in sib.find_all("li"):
                    bullets.append(li.get_text(separator=" ", strip=True))
            sib = sib.find_next_sibling()
        changes.append({"ability_or_stat": ability, "changes": bullets})
    return {"name": name, "context": context, "changes": changes}


def classify_change(changes):
    text = " ".join(b for c in changes for b in c["changes"]).lower()
    old_nums = re.findall(r"(\d+(?:\.\d+)?)\s*⇒", text)
    new_nums = re.findall(r"⇒\s*(\d+(?:\.\d+)?)", text)
    decreases = sum(1 for o, n in zip(old_nums, new_nums) if float(n) < float(o))
    increases = sum(1 for o, n in zip(old_nums, new_nums) if float(n) > float(o))
    if decreases > increases:
        return "nerf"
    elif increases > decreases:
        return "buff"
    if any(w in text for w in ["new", "added", "increased"]):
        return "buff"
    if any(w in text for w in ["removed", "decreased", "reduced"]):
        return "nerf"
    return "adjustment"


def parse_patch_page(html, url):
    soup = BeautifulSoup(html, "html.parser")

    title_tag     = soup.find("title")
    patch_version = title_tag.get_text(strip=True) if title_tag else "Unknown"

    date_str = ""
    time_tag = soup.find("time")
    if time_tag:
        date_str = time_tag.get("datetime", time_tag.get_text(strip=True))
    if not date_str:
        meta = soup.find("meta", attrs={"property": "article:published_time"})
        if meta:
            date_str = meta.get("content", "")
    if not date_str:
        m = re.search(r"patch-([\w-]+)-notes", url)
        date_str = m.group(1) if m else "unknown"
    date_str = date_str[:10]

    mid_patch = ""
    mp = soup.find(string=re.compile(r"Mid.Patch Updates", re.I))
    if mp and mp.parent:
        parts = [s.get_text(separator=" ", strip=True)
                 for s in mp.parent.find_next_siblings()][:5]
        mid_patch = " ".join(parts)

    # Per-subject change blocks (Champions / Items / Runes / Summoner Spells)
    sections = {}
    for div in soup.find_all("div"):
        h3 = div.find("h3", class_=re.compile("change-title"), recursive=False)
        if not h3:
            continue
        section_name = "General"
        for prev_h2 in div.find_all_previous("h2"):
            section_name = normalize_section(prev_h2.get_text(strip=True))
            break
        parsed = parse_change_block(div)
        if parsed:
            parsed["section"] = section_name
            sections.setdefault(section_name, []).append(parsed)

    # ── Bugfixes & QoL — one block per patch
    for h2 in soup.find_all("h2"):
        if normalize_section(h2.get_text(strip=True)) != "Bugfixes & QoL":
            continue

        bullets = []
        container = h2.parent 
        for sib in container.find_next_siblings():
            if sib.name == "header" and "header-primary" in (sib.get("class") or []):
                break
            for li in sib.find_all("li"):
                text = li.get_text(separator=" ", strip=True)
                if text:
                    bullets.append(text)
            if sib.name == "p":
                text = sib.get_text(strip=True)
                if text:
                    bullets.append(text)

        if bullets:
            existing = sections.get("Bugfixes & QoL", [])
            if existing:
                existing[0]["changes"][0]["changes"].extend(bullets)
            else:
                sections["Bugfixes & QoL"] = [{
                    "name":    "Bugfixes & QoL",
                    "context": f"Bug fixes and quality of life changes for {patch_version}",
                    "changes": [{"ability_or_stat": "", "changes": bullets}],
                    "section": "Bugfixes & QoL"
                }]

    return {
        "patch_version":     patch_version,
        "date":              date_str,
        "url":               url,
        "mid_patch_updates": mid_patch,
        "sections":          sections,
    }


# Build one RAG chunk
def build_rag_chunk(patch, section, entry, slug):
    all_bullets = [b for c in entry["changes"] for b in c["changes"]]
    summary = (
        f"{entry['name']} received changes in {patch['patch_version']}: "
        + "; ".join(all_bullets[:3])
    )
    lines = [
        f"Summary: {summary}",
        f"Patch: {patch['patch_version']}",
        f"Date: {patch['date']}",
        f"Section: {section}",
        f"Subject: {entry['name']}",
    ]
    if entry["context"]:
        lines.append(f"Context: {entry['context']}")
    for chg in entry["changes"]:
        if chg["ability_or_stat"]:
            lines.append(f"  {chg['ability_or_stat']}:")
        for b in chg["changes"]:
            lines.append(f"    - {b}")

    change_type = (
        "info" if section == "Bugfixes & QoL"
        else classify_change(entry["changes"])
    )

    chunk_id = (
        f"{slug}__{section}__{entry['name']}"
        .replace(" ", "_").lower()
        .replace("(", "").replace(")", "").replace("&", "and")
    )

    return {
        "id":            chunk_id,
        "patch_version": patch["patch_version"],
        "patch_date":    patch["date"],
        "patch_url":     patch["url"],
        "section":       section,
        "subject":       entry["name"],
        "change_type":   change_type,   # buff / nerf / adjustment / info
        "context":       entry["context"],
        "changes":       entry["changes"],
        "text":          "\n".join(lines),  
    }


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("=" * 60)
    print("Step 1: Discovering patch URLs...")
    print("=" * 60)
    all_urls = get_all_patch_urls()
    print(f"Total found: {len(all_urls)} patch URLs\n")
    all_urls.sort()

    print("=" * 60)
    print("Step 2: Scraping & parsing each patch...")
    print("=" * 60)

    all_patches = []
    all_chunks  = []

    for url in all_urls:
        slug = url.rstrip("/").split("/")[-1]
        print(f"  → {slug} … ", end="", flush=True)
        try:
            r = requests.get(url, headers=HEADERS, timeout=20)
        except Exception as e:
            print(f"ERROR ({e})")
            continue
        if r.status_code != 200:
            print(f"HTTP {r.status_code}, skipping")
            continue

        patch = parse_patch_page(r.text, url)
        all_patches.append(patch)

        n_entries = sum(len(v) for v in patch["sections"].values())
        print(f"✓  {patch['date']}  |  {n_entries} entries  |  {list(patch['sections'].keys())}")

        for section, entries in patch["sections"].items():
            for entry in entries:
                all_chunks.append(build_rag_chunk(patch, section, entry, slug))

        time.sleep(DELAY)

    with open(f"{OUTPUT_DIR}/all_patches.json", "w", encoding="utf-8") as f:
        json.dump(all_patches, f, ensure_ascii=False, indent=2)

    with open(f"{OUTPUT_DIR}/rag_chunks.jsonl", "w", encoding="utf-8") as f:
        for chunk in all_chunks:
            f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

    with open(f"{OUTPUT_DIR}/rag_chunks_pretty.json", "w", encoding="utf-8") as f:
        json.dump(all_chunks, f, ensure_ascii=False, indent=2)

    section_counts = Counter(c["section"] for c in all_chunks)
    print("\n" + "=" * 60)
    print(f"Done!  {len(all_patches)} patches  |  {len(all_chunks)} total RAG chunks")
    print("\nChunks per section:")
    for sec, count in sorted(section_counts.items()):
        print(f"   {sec:<35} {count} chunks")
    print(f"\nFiles saved to ./{OUTPUT_DIR}/")
    print(f"   all_patches.json        — full structured data")
    print(f"   rag_chunks.jsonl        — one line per chunk (embed 'text' field)")
    print(f"   rag_chunks_pretty.json  — same, indented for inspection")
    print("=" * 60)


if __name__ == "__main__":
    main()

Step 1: Discovering patch URLs...
  Probing patch-26-1 through patch-26-15 (both URL patterns)...
    ✓ patch-26-1 found: patch-26-1-notes
    ✓ patch-26-2 found: patch-26-2-notes
    ✓ patch-26-3 found: patch-26-3-notes
    ✓ patch-26-4 found: league-of-legends-patch-26-4-notes
    ✓ patch-26-5 found: league-of-legends-patch-26-5-notes
    ✓ patch-26-6 found: league-of-legends-patch-26-6-notes
    ✓ patch-26-7 found: league-of-legends-patch-26-7-notes
    ✓ patch-26-8 found: league-of-legends-patch-26-8-notes
    ✓ patch-26-9 found: league-of-legends-patch-26-9-notes
Total found: 9 patch URLs

Step 2: Scraping & parsing each patch...
  → league-of-legends-patch-26-4-notes … ✓  2026-02-18  |  34 entries  |  ['Champions', 'Bugfixes & QoL']
  → league-of-legends-patch-26-5-notes … ✓  2026-03-03  |  17 entries  |  ['Champions', 'Items', 'Bugfixes & QoL']
  → league-of-legends-patch-26-6-notes … ✓  2026-03-17  |  13 entries  |  ['Champions', 'Items', 'Bugfixes & QoL']
  → league-of-legends